## INIT

In [0]:
import sys
import os
sys.path.append(os.path.abspath('../../'))
from utils.logger_utils import get_project_logger

logger = get_project_logger("Gold_Dimension_ownership")
logger.info("--- Starting Gold Layer: Creating dim_ownership ---")

## Creating a gold dimension view

In [0]:
try:
    logger.info("Step 1/2: Running SQL to create dim_ownership view")
    
    sql_query = """
    CREATE OR REPLACE VIEW transport_lakehouse.gold.dim_ownership AS
    SELECT
    ROW_NUMBER() OVER (ORDER BY ownership) AS ownership_key,
    ownership
    FROM (
    SELECT ownership FROM transport_lakehouse.silver.silver_vehicles_private
    UNION
    SELECT ownership FROM transport_lakehouse.silver.silver_vehicles_electric
    UNION
    SELECT ownership FROM transport_lakehouse.silver.silver_vehicles_two_wheeled
    
    ) t
    WHERE ownership IS NOT NULL
    """
    
    spark.sql(sql_query)
    logger.info("Step 1/2: View created successfully.")

    logger.info("Step 2/2: Performing Quality Check (Row Count)")
    dim_count = spark.table("transport_lakehouse.gold.dim_ownership").count()
    logger.info(f"Quality Check Passed: dim_ownership contains {dim_count} unique ownerships.")

except Exception as e:
    logger.error(f"Failed to create Gold Dimension: {str(e)}")
    raise e

logger.info("--- Gold dim_ownership Process Completed ---")
